<a href="https://colab.research.google.com/github/Sayak-coder/HateSpeech-detection/blob/main/Subtask_(B)_Hate_Speech_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [2]:
df_en = pd.read_csv('/content/StereoQueerEval_EN_training.tsv',sep='\t')
df_en.head()

,StereoQueerEval_id,yt_title,yt_description,yt_comment,stereotype,hate_speech,target
0,training_EN_0001,[CHANNEL]: Lesbian denied communion at funeral,[CHANNEL] speaks to Barbara Johnson who says a...,She's living in unrepentant defiance to God's ...,yes,yes_explicit,individual_l
1,training_EN_0002,GOP Congresswoman uses anti-trans slur in Hous...,Rep. Nancy Mace (R-SC) was challenged by Rep. ...,Rep Mace: Keep on being dedicated to truth and...,yes,yes_implicit,group_t
2,training_EN_0003,Kevin Smith defends his gay brother,"Kevin Smith supports gay rights, and defends h...","We all need Jesus, He said no homosexual shall...",yes,yes_implicit,"group_l,g"
3,training_EN_0004,Judge removes child from gay foster parents,A Utah judge removed a foster child from a les...,So can two black men adopt a little white girl...,yes,yes_implicit,group_l
4,training_EN_0005,Chechen leader Ramzan Kadyrov questioned on 'g...,Chechnya's leader Ramzan Kadyrov is questioned...,Of course west and US will hate him,yes,yes_implicit,group_g


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [5]:
df = df_en

### EXperiment A
comment ony as input

In [ ]:
df['hate_speech'].value_counts()

,count
hate_speech,
no,1407
yes_explicit,810
yes_implicit,772


In [ ]:
df.isnull().sum()

,0
StereoQueerEval_id,0
yt_title,0
yt_description,0
yt_comment,0
stereotype,0
hate_speech,0
target,0


In [ ]:
X = list(df['yt_comment'])
y = df['hate_speech'].map({'no':0, 'yes_explicit':1, 'yes_implicit':2}).tolist()

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=45)
X_val,X_test,y_val,y_test = train_test_split(X_test,y_test,test_size=0.5,random_state=45)

In [ ]:
!pip install transformers

### sentiment-roberta-large-english

In [ ]:
from transformers import AutoTokenizer

MODEL = f"siebert/sentiment-roberta-large-english"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [ ]:
train_encodings = tokenizer(X_train, truncation= True, padding=True, max_length=128)
val_encodings = tokenizer(X_val, truncation= True, padding=True, max_length=128)
test_encodings = tokenizer(X_test, truncation= True, padding=True, max_length=128)

In [ ]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
val_dataset = TextDataset(val_encodings, y_val)
test_dataset = TextDataset(test_encodings, y_test)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer

# Explicitly set num_labels=3 to match our targets (0, 1, 2) and ignore mismatched classification head sizes
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True
).to(device)

trainer_RoBERTa = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer_RoBERTa.train()

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.42GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: siebert/sentiment-roberta-large-english
Key                        | Status   |                                                                                         
---------------------------+----------+-----------------------------------------------------------------------------------------
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2, 1024]) vs model:torch.Size([3, 1024])
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([3])            

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Step,Training Loss
10,1.139575
20,1.098564
30,1.037846
40,1.099744
50,0.963817
60,1.040439
70,1.078754
80,0.996895
90,1.035690
100,1.050679


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=600, training_loss=0.833560410340627, metrics={'train_runtime': 781.9372, 'train_samples_per_second': 12.231, 'train_steps_per_second': 0.767, 'total_flos': 2228255418498048.0, 'train_loss': 0.833560410340627, 'epoch': 4.0})

In [ ]:
trainer_RoBERTa.evaluate(val_dataset)

Training Loss,Validation Loss,Step
0.644711,0.917291,600


{'eval_loss': 0.9172910451889038}

In [ ]:
y_pred = np.argmax(trainer_RoBERTa.predict(test_dataset).predictions, axis=-1)

In [ ]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.77      0.70      0.73       142
           1       0.60      0.63      0.61        81
           2       0.40      0.45      0.42        76

    accuracy                           0.62       299
   macro avg       0.59      0.59      0.59       299
weighted avg       0.63      0.62      0.62       299



###BERT

In [ ]:
from transformers import  AutoTokenizer

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
train_encodings = tokenizer(X_train, truncation= True, padding=True, max_length=128)
val_encodings = tokenizer(X_val, truncation= True, padding=True, max_length=128)
test_encodings = tokenizer(X_test, truncation= True, padding=True, max_length=128)

In [ ]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
val_dataset = TextDataset(val_encodings, y_val)
test_dataset = TextDataset(test_encodings, y_test)

In [ ]:
from transformers import AutoModelForSequenceClassification,Trainer,TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=60,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=10,
)

In [ ]:
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)

trainer_BERT = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer_BERT.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Step,Training Loss
10,1.906377
20,1.609857
30,1.258582
40,1.143013
50,1.104092
60,1.045948
70,1.197693
80,1.063005
90,1.009052
100,0.999393


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1196, training_loss=0.6589065104423957, metrics={'train_runtime': 330.0837, 'train_samples_per_second': 28.974, 'train_steps_per_second': 3.623, 'total_flos': 629115478631424.0, 'train_loss': 0.6589065104423957, 'epoch': 4.0})

In [ ]:
from accelerate.state import PartialState
from transformers import Trainer

# Reset and properly re-instantiate the global accelerator state
PartialState._reset_state()
_ = PartialState()

# Re-create the trainer so it uses the clean accelerator state
trainer_BERT = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer_BERT.evaluate(val_dataset)

Training Loss,Validation Loss,Step
No log,1.492645,0


{'eval_loss': 1.4926449060440063}

In [ ]:
y_pred = np.argmax(trainer_BERT.predict(test_dataset).predictions, axis=-1)

In [ ]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.65      0.70      0.67       142
           1       0.68      0.54      0.60        81
           2       0.38      0.41      0.39        76

    accuracy                           0.58       299
   macro avg       0.57      0.55      0.56       299
weighted avg       0.59      0.58      0.58       299



###cardiffnlp/twitter-xlm-roberta

In [ ]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-xlm-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [ ]:
train_encodings = tokenizer(X_train, truncation= True, padding=True, max_length=128)
val_encodings = tokenizer(X_val, truncation= True, padding=True, max_length=128)
test_encodings = tokenizer(X_test, truncation= True, padding=True, max_length=128)

In [ ]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
val_dataset = TextDataset(val_encodings, y_val)
test_dataset = TextDataset(test_encodings, y_test)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=60,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=10,
    max_grad_norm=1.0
)

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer


model = AutoModelForSequenceClassification.from_pretrained(MODEL).to(device)

trainer_XLMRoBERTa = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer_XLMRoBERTa.train()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Step,Training Loss
10,1.639462
20,1.431224
30,1.215437
40,1.113275
50,1.102514
60,1.039256
70,1.160563
80,1.046338
90,1.007897
100,1.008898


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss
10,1.639462
20,1.431224
30,1.215437
40,1.113275
50,1.102514
60,1.039256
70,1.160563
80,1.046338
90,1.007897
100,1.008898


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1495, training_loss=0.58080667432734, metrics={'train_runtime': 489.6446, 'train_samples_per_second': 24.416, 'train_steps_per_second': 3.053, 'total_flos': 786380227234560.0, 'train_loss': 0.58080667432734, 'epoch': 5.0})

In [ ]:
trainer_XLMRoBERTa.evaluate(val_dataset)

Training Loss,Validation Loss,Step
0.375330,1.516505,1495


{'eval_loss': 1.5165045261383057}

In [ ]:
y_pred = np.argmax(trainer_XLMRoBERTa.predict(test_dataset).predictions, axis=-1)

In [ ]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.71      0.75      0.73       142
           1       0.67      0.59      0.63        81
           2       0.46      0.47      0.47        76

    accuracy                           0.64       299
   macro avg       0.61      0.60      0.61       299
weighted avg       0.64      0.64      0.63       299



### DistilBERT

In [ ]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [ ]:
train_encodings = tokenizer(X_train, truncation= True, padding=True, max_length=128)
val_encodings = tokenizer(X_val, truncation= True, padding=True, max_length=128)
test_encodings = tokenizer(X_test, truncation= True, padding=True, max_length=128)

In [ ]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
val_dataset = TextDataset(val_encodings, y_val)
test_dataset = TextDataset(test_encodings, y_test)

In [ ]:
from transformers import (
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=60,
    weight_decay=0.01,
    logging_steps=10,
)

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english", num_labels=3, ignore_mismatched_sizes=True).to(device)

trainer_distilBERT = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer_distilBERT.train()

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([3])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2, 768]) vs model:torch.Size([3, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Step,Training Loss
10,1.061266
20,1.055131
30,1.073817
40,1.077763
50,1.067053
60,1.031613
70,1.139366
80,1.019462
90,0.981449
100,0.925892


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=598, training_loss=0.8509343595409075, metrics={'train_runtime': 118.3651, 'train_samples_per_second': 40.4, 'train_steps_per_second': 5.052, 'total_flos': 158367599304192.0, 'train_loss': 0.8509343595409075, 'epoch': 2.0})

In [ ]:
trainer_distilBERT.evaluate(val_dataset)

Training Loss,Validation Loss,Step
0.751573,0.907526,598


{'eval_loss': 0.907526433467865}

In [ ]:
y_pred = np.argmax(trainer_distilBERT.predict(test_dataset).predictions, axis=-1)

In [ ]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.68      0.68      0.68       142
           1       0.51      0.47      0.49        81
           2       0.32      0.34      0.33        76

    accuracy                           0.54       299
   macro avg       0.50      0.50      0.50       299
weighted avg       0.54      0.54      0.54       299



### EXperiment B
comment + title as input

In [ ]:
df.columns

Index(['StereoQueerEval_id', 'yt_title', 'yt_description', 'yt_comment',
       'stereotype', 'hate_speech', 'target'],
      dtype='object')

In [ ]:
X = (df.drop(['StereoQueerEval_id', 'yt_description', 'stereotype', 'target', 'hate_speech'], axis=1))
y = df['hate_speech'].map({'no':0, 'yes_explicit':1, 'yes_implicit':2})

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=45)
X_val,X_test,y_val,y_test = train_test_split(X_test,y_test,test_size=0.5,random_state=45)

In [ ]:
def build_inputs(d):
    comments = d['yt_comment'].tolist()
    title = d['yt_title'].fillna("")
    ctx = title
    return comments, ctx.tolist()

def encode(d, tokenizer, max_length=256):
    comments, ctx = build_inputs(d)
    return tokenizer(comments, ctx, truncation="longest_first", max_length=max_length)

In [ ]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

###cardiffnlp/twitter-roberta-base-hate-latest

In [ ]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-roberta-base-hate-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/888 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [ ]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [ ]:
from transformers import AutoModelForSequenceClassification

MODEL = "cardiffnlp/twitter-roberta-base-hate-latest"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "explicit", 2: "implicit"},
    label2id={"no": 0, "explicit": 1, "implicit": 2},
).to(device)

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  499MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-hate-latest
Key                        | Status   |                                                                                       
---------------------------+----------+---------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([3])          
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2, 768]) vs model:torch.Size([3, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [ ]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_twitter_roberta = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_twitter_roberta.train()

Step,Training Loss
10,1.069120
20,1.063781
30,1.019820
40,1.032017
50,0.968367
60,0.996225
70,0.995352
80,0.918964
90,0.932386
100,0.982443


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=600, training_loss=0.6929933492342631, metrics={'train_runtime': 292.9977, 'train_samples_per_second': 32.642, 'train_steps_per_second': 2.048, 'total_flos': 790767849032532.0, 'train_loss': 0.6929933492342631, 'epoch': 4.0})

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_twitter_roberta.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","explicit","implicit"], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.717     0.768     0.741       142
    explicit      0.654     0.630     0.642        81
    implicit      0.464     0.421     0.441        76

    accuracy                          0.642       299
   macro avg      0.612     0.606     0.608       299
weighted avg      0.636     0.642     0.638       299

[[109  14  19]
 [ 12  51  18]
 [ 31  13  32]]


###deberta-v3-large

In [ ]:
from transformers import AutoTokenizer

MODEL = f"microsoft/deberta-v3-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

In [ ]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    learning_rate=5e-6,
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=10
)

In [ ]:
from transformers import AutoModelForSequenceClassification

MODEL = f"microsoft/deberta-v3-large"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "explicit", 2: "implicit"},
    label2id={"no": 0, "explicit": 1, "implicit": 2},
).to(device)

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  874MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifie

In [ ]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_deberta = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_deberta.train()

Step,Training Loss
10,1.086566
20,1.077497
30,1.051424
40,1.109023
50,1.039161
60,1.114497
70,1.088529
80,1.032787
90,1.089500
100,1.054951


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=750, training_loss=1.0699276555379231, metrics={'train_runtime': 434.0927, 'train_samples_per_second': 27.54, 'train_steps_per_second': 1.728, 'total_flos': 3391367167982934.0, 'train_loss': 1.0699276555379231, 'epoch': 5.0})

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_deberta.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","explicit","implicit"], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.475     1.000     0.644       142
    explicit      0.000     0.000     0.000        81
    implicit      0.000     0.000     0.000        76

    accuracy                          0.475       299
   macro avg      0.158     0.333     0.215       299
weighted avg      0.226     0.475     0.306       299

[[142   0   0]
 [ 81   0   0]
 [ 76   0   0]]


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


###microsoft/mdeberta-v3-base

In [ ]:
from transformers import AutoTokenizer

MODEL = f"microsoft/mdeberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [ ]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=75,
    learning_rate=1e-5,
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=10
)

In [ ]:
from transformers import AutoModelForSequenceClassification

MODEL = f"microsoft/mdeberta-v3-base"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "explicit", 2: "implicit"},
    label2id={"no": 0, "explicit": 1, "implicit": 2},
).to(device)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
mask_predictions.classifier.bias           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
classifier.bias     

In [ ]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_mdeberta_v3_base = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_mdeberta_v3_base.train()

Step,Training Loss
10,1.114492
20,1.097415
30,1.053639
40,1.111021
50,1.055719
60,1.077093
70,1.127098
80,1.101421
90,1.098145
100,1.061166


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=750, training_loss=1.0727039362589519, metrics={'train_runtime': 205.3311, 'train_samples_per_second': 58.223, 'train_steps_per_second': 3.653, 'total_flos': 1118410875266490.0, 'train_loss': 1.0727039362589519, 'epoch': 5.0})

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_mdeberta_v3_base.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","explicit","implicit"], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.475     1.000     0.644       142
    explicit      0.000     0.000     0.000        81
    implicit      0.000     0.000     0.000        76

    accuracy                          0.475       299
   macro avg      0.158     0.333     0.215       299
weighted avg      0.226     0.475     0.306       299

[[142   0   0]
 [ 81   0   0]
 [ 76   0   0]]


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#Experiment 3
comment + title + description as input


In [6]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [12]:
df = df_en

In [14]:
df['hate_speech'].unique()

array(['yes_explicit', 'yes_implicit', 'no'], dtype=object)

In [18]:
X = (df.drop(['StereoQueerEval_id', 'stereotype', 'target', 'hate_speech'], axis=1))
y = df['hate_speech'].map({"no": 0, "yes_explicit": 1, "yes_implicit": 2})

In [19]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=45)
X_val,X_test,y_val,y_test = train_test_split(X_test,y_test,test_size=0.5,random_state=45)

In [20]:
def build_inputs(d):
    comments = d['yt_comment'].tolist()
    title = d['yt_title'].fillna("")
    desc  = d['yt_description'].fillna("").str[:300]        # trim long descriptions
    ctx = (title + " . " + desc)
    return comments, ctx.tolist()

def encode(d, tokenizer, max_length=256):
    comments, ctx = build_inputs(d)
    return tokenizer(comments, ctx, truncation="longest_first", max_length=max_length)

In [21]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [22]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [23]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [26]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [27]:
from transformers import DistilBertForSequenceClassification

MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"

# Correctly load the pre-trained model and configure it in one step
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes_explicit", 2:"yes_implicit"},
    label2id={"no": 0, "yes_explicit": 1, "yes_implicit": 2}
).to(device)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([3])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2, 768]) vs model:torch.Size([3, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [28]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_distilbert = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_distilbert.train()

Step,Training Loss
10,1.116259
20,1.106356
30,1.128748
40,1.097474
50,1.120388
60,1.042309
70,1.120267
80,1.065856
90,1.015281
100,0.993144


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1196, training_loss=0.8416397810779687, metrics={'train_runtime': 196.5529, 'train_samples_per_second': 48.659, 'train_steps_per_second': 6.085, 'total_flos': 438688361241090.0, 'train_loss': 0.8416397810779687, 'epoch': 4.0})

In [30]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_distilbert.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes_explicit","yes_implicit"], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.642     0.683     0.662       142
yes_explicit      0.494     0.469     0.481        81
yes_implicit      0.408     0.382     0.395        76

    accuracy                          0.548       299
   macro avg      0.515     0.511     0.513       299
weighted avg      0.543     0.548     0.545       299

[[97 22 23]
 [24 38 19]
 [30 17 29]]


###cardiffnlp/twitter-roberta-base-hate-latest

In [31]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-roberta-base-hate-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/888 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [32]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [33]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [34]:
from transformers import AutoModelForSequenceClassification

MODEL = "cardiffnlp/twitter-roberta-base-hate-latest"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=2,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes_explicit", 2:"yes_implicit"},
    label2id={"no": 0, "yes_explicit": 1, "yes_implicit": 2},
).to(device)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  499MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-hate-latest
Key                        | Status   |                                                                                       
---------------------------+----------+---------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([3])          
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2, 768]) vs model:torch.Size([3, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [35]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_twitter_roberta = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_twitter_roberta.train()

Step,Training Loss
10,1.074623
20,1.063516
30,1.036381
40,1.058175
50,0.997774
60,1.018542
70,1.039603
80,0.973520
90,0.958107
100,0.948493


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=600, training_loss=0.6918462113539378, metrics={'train_runtime': 407.7897, 'train_samples_per_second': 23.453, 'train_steps_per_second': 1.471, 'total_flos': 1026707611872600.0, 'train_loss': 0.6918462113539378, 'epoch': 4.0})

In [36]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_twitter_roberta.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes_explicit","yes_implicit"], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.752     0.768     0.760       142
yes_explicit      0.655     0.679     0.667        81
yes_implicit      0.486     0.447     0.466        76

    accuracy                          0.662       299
   macro avg      0.631     0.631     0.631       299
weighted avg      0.658     0.662     0.660       299

[[109  16  17]
 [  7  55  19]
 [ 29  13  34]]
